<a href="https://colab.research.google.com/github/AhmadZahran1011/BigData26_A_2411532004_AhmadZahran/blob/praktikum2/praktikum2/BD_A_P02_2411532004_AhmadZahran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 47.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")         # ikuti dialog izin yang muncul

import os
DIR_KERJA = "/content/data"                                 # sementara, cepat
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"    # permanen
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Mounted at /content/drive
['transaksi_bersih.csv']


k-1

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random

k-2

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar}",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()

    if random.random() < 0.2:
        kategori = kategori.upper()

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))


Jumlah baris: 515


k-3

In [ ]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [ ]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

k-4

In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


k-5

In [ ]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})


In [ ]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [ ]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [ ]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

k-6

In [ ]:
df.to_csv(f"{DIR_SIMPAN}/transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


STUDI KASUS

Nomor 1: Perbedaan angka terjadi karena data IT 515 baris merupakan data mentah yang belum diolah, sedangkan data Finance 490 baris adalah data bersih setelah membuang 5 baris transaksi duplikat dan 20 baris data yang kehilangan informasi penting seperti nama pelanggan dan metode pembayaran.

Nomor 2: Angka 490 baris "lebih benar" karena mematuhi prinsip Veracity (kualitas dan keandalan data) dengan mengeliminasi noise dan data tidak valid yang berpotensi menyebabkan perhitungan ganda (overcounting) pada laporan keuangan.

 Nomor 3: Kolom rating dibiarkan memiliki missing value karena pemberian ulasan bersifat opsional dari pembeli sehingga transaksinya tetap sah secara finansial, dan pengisian nilai buatan hanya akan merusak keakuratan perhitungan rata-rata rating sebenarnya.  

Latihan-1

In [ ]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows_7 = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar}",
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()

    if random.random() < 0.2:
        kategori = kategori.upper()

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows_7.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df_7 = pd.DataFrame(rows_7)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df_7.sample(frac=frac, random_state=SEED).index
    df_7.loc[idx, col] = np.nan

dup_rows = df_7.sample(n=15, random_state=SEED)
df_7 = pd.concat([df_7, dup_rows], ignore_index=True)
df_7 = df_7.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_7.to_csv("transaksi_mentah_seed7.csv", index=False)
len_mentah_7 = len(df_7)

# Pembersihan Data SEED = 7
df_7 = df_7.dropna(subset=["customer_name", "payment_method"])
df_7["shipping_city"] = df_7["shipping_city"].fillna("Tidak Diketahui")
df_7 = df_7.drop_duplicates()

for col in ["category", "payment_method", "shipping_city"]:
    df_7[col] = df_7[col].astype("string").str.strip().str.title()

df_7["payment_method"] = df_7["payment_method"].replace({"Cod": "COD"})
df_7["price"] = df_7["price"].apply(bersihkan_harga)
df_7["transaction_date"] = df_7["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
df_7["quantity"] = df_7["quantity"].astype(int)
df_7["price"] = df_7["price"].astype(float)

df_7.to_csv(f"{DIR_SIMPAN}/transaksi_bersih_seed7.csv", index=False)
len_bersih_7 = len(df_7)

print(f"SEED = 42 -> transaksi_mentah: 515 baris | transaksi_bersih: 490 baris")
print(f"SEED = 7  -> transaksi_mentah: {len_mentah_7} baris | transaksi_bersih: {len_bersih_7} baris")

SEED = 42 -> transaksi_mentah: 515 baris | transaksi_bersih: 490 baris
SEED = 7  -> transaksi_mentah: 515 baris | transaksi_bersih: 490 baris


Latihan 2

In [ ]:
df["is_valid_price"] = df["price"] > 0

invalid_prices = df[~df["is_valid_price"]]
print("Jumlah harga tidak valid (price <= 0):", len(invalid_prices))

if len(invalid_prices) > 0:
    print(invalid_prices[["transaction_id", "price", "is_valid_price"]])

Jumlah harga tidak valid (price <= 0): 0


Latihan 2

In [ ]:
jumlah_per_kategori = df["category"].value_counts()
print(jumlah_per_kategori)

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64
